# Vanilla BiRNN V3 - Maximum Optimization (Stable)

This notebook implements a **strictly vanilla** Bidirectional RNN with all possible optimizations, tuned for stability:
1. **Tanh Activation** (Stable, preventing explosion)
2. **Deeper & Wider** (4 layers, 512 hidden units)
3. **Residual Connections** between RNN layers
4. **Layer Normalization** after RNN
5. **Dropout** for regularization
6. **Orthogonal Initialization** (Critical for Tanh RNNs)
7. **CosineAnnealingWarmRestarts** scheduler
8. **30 Epochs** training

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

class Config:
    seed = 42
    device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
    batch_size = 32  # Smaller batch for larger model
    embed_dim = 128
    hidden_dim = 512  # Increased
    num_layers = 4    # Increased
    dropout = 0.3
    learning_rate = 0.0005 # Lowered for stability
    epochs = 30       # Increased
    
    train_path = "./train.csv"
    test_path = "./test.csv"
    
    trackio_project = "25-t3-nppe2"
    trackio_space_id = "manyakaistha/dlgenai-nppe"
    trackio_group = "vanilla_birnn"

cfg = Config()
print(f"Device: {cfg.device}")

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

os.environ["HUGGINGFACEHUB_API_TOKEN"] = "hf_placeholder_token_for_reproducibility"
os.environ["TRACKIO_SPACE_ID"] = cfg.trackio_space_id

def maybe_init_trackio():
    try:
        import trackio
        token = os.environ.get("HUGGINGFACEHUB_API_TOKEN") or os.environ.get("HF_TOKEN")
        if cfg.trackio_space_id and token:
            trackio.init(project=cfg.trackio_project, space_id=cfg.trackio_space_id, group=cfg.trackio_group)
            trackio.log({"status/started": 1})
            print("✓ TrackIO initialized")
            return trackio
    except Exception as e:
        print(f"TrackIO init failed: {e}")
    return None

trk = maybe_init_trackio()

## Data Preprocessing

In [ ]:
PAD_TOKEN = 0

AA_CHARS = ['<PAD>', '<UNK>', '*', 'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
AA_TO_ID = {c: i for i, c in enumerate(AA_CHARS)}
ID_TO_AA = {i: c for c, i in AA_TO_ID.items()}

Q8_CHARS = ['<PAD>'] + sorted(['B', 'C', 'E', 'G', 'H', 'I', 'S', 'T'])
Q8_TO_ID = {c: i for i, c in enumerate(Q8_CHARS)}
ID_TO_Q8 = {i: c for c, i in Q8_TO_ID.items()}

Q3_CHARS = ['<PAD>'] + sorted(['C', 'E', 'H'])
Q3_TO_ID = {c: i for i, c in enumerate(Q3_CHARS)}
ID_TO_Q3 = {i: c for c, i in Q3_TO_ID.items()}

class ProteinDataset(Dataset):
    def __init__(self, csv_path, mode='train'):
        self.data = pd.read_csv(csv_path)
        self.mode = mode
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        seq_ids = [AA_TO_ID.get(c, AA_TO_ID['<UNK>']) for c in row['seq']]
        item = {'seq': torch.tensor(seq_ids, dtype=torch.long), 'id': row['id']}
        if self.mode == 'train':
            item['q8'] = torch.tensor([Q8_TO_ID.get(c, 0) for c in row['sst8']], dtype=torch.long)
            item['q3'] = torch.tensor([Q3_TO_ID.get(c, 0) for c in row['sst3']], dtype=torch.long)
        return item

def collate_fn(batch):
    batch.sort(key=lambda x: len(x['seq']), reverse=True)
    seqs = [item['seq'] for item in batch]
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=PAD_TOKEN)
    out = {'seq': seqs_padded, 'ids': [item['id'] for item in batch], 'lengths': lengths}
    if 'q8' in batch[0]:
        out['q8'] = pad_sequence([item['q8'] for item in batch], batch_first=True, padding_value=PAD_TOKEN)
        out['q3'] = pad_sequence([item['q3'] for item in batch], batch_first=True, padding_value=PAD_TOKEN)
    return out

train_ds = ProteinDataset(cfg.train_path, mode='train')
val_size = int(0.1 * len(train_ds))
train_subset, val_subset = random_split(train_ds, [len(train_ds) - val_size, val_size], generator=torch.Generator().manual_seed(cfg.seed))
train_loader = DataLoader(train_subset, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_subset, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)
print(f"Train: {len(train_subset)}, Val: {len(val_subset)}")

### Class Weights

In [ ]:
df_train = pd.read_csv(cfg.train_path)
all_q8 = [Q8_TO_ID[c] for s in df_train['sst8'] for c in s]
all_q3 = [Q3_TO_ID[c] for s in df_train['sst3'] for c in s]

classes_q8 = np.arange(1, len(Q8_CHARS))
classes_q3 = np.arange(1, len(Q3_CHARS))

weights_q8_valid = compute_class_weight('balanced', classes=classes_q8, y=all_q8)
weights_q3_valid = compute_class_weight('balanced', classes=classes_q3, y=all_q3)

weights_q8 = np.zeros(len(Q8_CHARS))
weights_q8[1:] = weights_q8_valid
weights_q3 = np.zeros(len(Q3_CHARS))
weights_q3[1:] = weights_q3_valid

weight_q8 = torch.tensor(weights_q8, dtype=torch.float32).to(cfg.device)
weight_q3 = torch.tensor(weights_q3, dtype=torch.float32).to(cfg.device)
print(f"Q8 Weights: {weights_q8}")

## Model: Residual Vanilla BiRNN (Stable Tanh)

In [ ]:
class ResidualBiRNN(nn.Module):
    """Vanilla BiRNN with Residual Connections between layers."""
    
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, num_q8, num_q3, dropout=0.3):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_TOKEN)
        
        # First RNN layer (input is embed_dim)
        self.rnn_layers = nn.ModuleList()
        self.rnn_layers.append(
            nn.RNN(embed_dim, hidden_dim, num_layers=1, batch_first=True, 
                   bidirectional=True, nonlinearity='tanh')
        )
        
        # Subsequent RNN layers (input is hidden_dim * 2 due to bidirectional)
        for _ in range(num_layers - 1):
            self.rnn_layers.append(
                nn.RNN(hidden_dim * 2, hidden_dim, num_layers=1, batch_first=True,
                       bidirectional=True, nonlinearity='tanh')
            )
        
        # Layer Norm for each layer output
        self.layer_norms = nn.ModuleList([nn.LayerNorm(hidden_dim * 2) for _ in range(num_layers)])
        
        self.dropout = nn.Dropout(dropout)
        
        # Heads
        self.q8_head = nn.Linear(hidden_dim * 2, num_q8)
        self.q3_head = nn.Linear(hidden_dim * 2, num_q3)
        
        self._init_weights()
    
    def _init_weights(self):
        # Orthogonal init for Tanh RNNs
        for rnn in self.rnn_layers:
            for name, param in rnn.named_parameters():
                if 'weight_hh' in name:
                    nn.init.orthogonal_(param)
                elif 'weight_ih' in name:
                    nn.init.xavier_uniform_(param)
                elif 'bias' in name:
                    nn.init.constant_(param, 0.0)
        
        nn.init.xavier_uniform_(self.q8_head.weight)
        nn.init.xavier_uniform_(self.q3_head.weight)
    
    def forward(self, x, lengths):
        # x: [B, L]
        embed = self.embedding(x)  # [B, L, E]
        
        # First layer
        packed = pack_padded_sequence(embed, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.rnn_layers[0](packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        out = self.layer_norms[0](out)
        out = self.dropout(out)
        
        # Subsequent layers with residual
        for i in range(1, self.num_layers):
            residual = out
            packed = pack_padded_sequence(out, lengths.cpu(), batch_first=True, enforce_sorted=False)
            out, _ = self.rnn_layers[i](packed)
            out, _ = pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
            out = self.layer_norms[i](out)
            out = out + residual  # Residual Connection
            out = self.dropout(out)
        
        q8_logits = self.q8_head(out)
        q3_logits = self.q3_head(out)
        
        return q8_logits, q3_logits

model = ResidualBiRNN(
    vocab_size=len(AA_TO_ID),
    embed_dim=cfg.embed_dim,
    hidden_dim=cfg.hidden_dim,
    num_layers=cfg.num_layers,
    num_q8=len(Q8_TO_ID),
    num_q3=len(Q3_TO_ID),
    dropout=cfg.dropout
).to(cfg.device)

print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=cfg.learning_rate)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

criterion_q8 = nn.CrossEntropyLoss(weight=weight_q8, ignore_index=PAD_TOKEN)
criterion_q3 = nn.CrossEntropyLoss(weight=weight_q3, ignore_index=PAD_TOKEN)

def calc_f1(preds, targets, pad_idx=0):
    preds_flat = preds.argmax(dim=-1).view(-1).cpu().numpy()
    targets_flat = targets.view(-1).cpu().numpy()
    mask = targets_flat != pad_idx
    return f1_score(targets_flat[mask], preds_flat[mask], average='macro')

def train_epoch(model, loader, optimizer, device, epoch):
    model.train()
    total_loss, total_f1_q8, total_f1_q3, count = 0, 0, 0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}")
    for batch in pbar:
        seq = batch['seq'].to(device)
        lengths = batch['lengths']
        q8_tgt = batch['q8'].to(device)
        q3_tgt = batch['q3'].to(device)
        
        optimizer.zero_grad()
        q8_out, q3_out = model(seq, lengths)
        
        loss_q8 = criterion_q8(q8_out.view(-1, len(Q8_TO_ID)), q8_tgt.view(-1))
        loss_q3 = criterion_q3(q3_out.view(-1, len(Q3_TO_ID)), q3_tgt.view(-1))
        loss = loss_q8 + loss_q3
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5) # Lower clip
        optimizer.step()
        
        total_loss += loss.item()
        total_f1_q8 += calc_f1(q8_out.detach(), q8_tgt)
        total_f1_q3 += calc_f1(q3_out.detach(), q3_tgt)
        count += 1
        pbar.set_postfix({'loss': loss.item()})
        
        if trk:
            trk.log({'train/loss': loss.item()})
    
    return total_loss/count, total_f1_q8/count, total_f1_q3/count

def evaluate(model, loader, device):
    model.eval()
    total_loss, total_f1_q8, total_f1_q3, count = 0, 0, 0, 0
    with torch.no_grad():
        for batch in loader:
            seq = batch['seq'].to(device)
            lengths = batch['lengths']
            q8_tgt = batch['q8'].to(device)
            q3_tgt = batch['q3'].to(device)
            
            q8_out, q3_out = model(seq, lengths)
            loss = criterion_q8(q8_out.view(-1, len(Q8_TO_ID)), q8_tgt.view(-1)) + \
                   criterion_q3(q3_out.view(-1, len(Q3_TO_ID)), q3_tgt.view(-1))
            
            total_loss += loss.item()
            total_f1_q8 += calc_f1(q8_out, q8_tgt)
            total_f1_q3 += calc_f1(q3_out, q3_tgt)
            count += 1
    return total_loss/count, total_f1_q8/count, total_f1_q3/count

best_score = 0.0
for epoch in range(cfg.epochs):
    train_loss, train_f1_8, train_f1_3 = train_epoch(model, train_loader, optimizer, cfg.device, epoch)
    val_loss, val_f1_8, val_f1_3 = evaluate(model, val_loader, cfg.device)
    
    val_score = (2 * val_f1_8 * val_f1_3) / (val_f1_8 + val_f1_3 + 1e-8)
    scheduler.step()
    
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f} | Val Score={val_score:.4f} (Q8={val_f1_8:.4f}, Q3={val_f1_3:.4f})")
    
    if trk:
        trk.log({'val/score': val_score, 'val/loss': val_loss})
    
    if val_score > best_score:
        best_score = val_score
        torch.save(model.state_dict(), 'best_model_v3.pth')
        print("  --> Best model saved")

print(f"\nTraining Complete. Best Val Score: {best_score:.4f}")

## Inference

In [ ]:
test_ds = ProteinDataset(cfg.test_path, mode='test')
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

model.load_state_dict(torch.load('best_model_v3.pth'))
model.eval()
print("Loaded best model.")

results = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference"):
        seq = batch['seq'].to(cfg.device)
        lengths = batch['lengths']
        ids = batch['ids']
        
        q8_out, q3_out = model(seq, lengths)
        q8_preds = q8_out.argmax(dim=-1).cpu().numpy()
        q3_preds = q3_out.argmax(dim=-1).cpu().numpy()
        
        for i, id_ in enumerate(ids):
            length = lengths[i].item()
            q8_str = ''.join([ID_TO_Q8[idx] for idx in q8_preds[i, :length]])
            q3_str = ''.join([ID_TO_Q3[idx] for idx in q3_preds[i, :length]])
            results.append({'id': id_, 'sst8': q8_str, 'sst3': q3_str})

df_sub = pd.DataFrame(results)
df_sub.to_csv('submission_v3.csv', index=False)
print("Saved submission_v3.csv")
df_sub.head()